In [12]:
import pandas as pd 
import matplotlib.pyplot as plt
import json

In [13]:
df = pd.read_csv("../data/scrape/honestdoor_listing_details.csv")

# Remove junk columns
df = df.loc[:, ~df.columns.str.contains("^Unnamed")]

display(df.head())

,Property URL,id,assessmentClass,zoning,bathroomsTotal,bathroomsTotalEst,bedroomsTotal,bedroomsTotalEst,livingArea,livingAreaEst,...,houseStyle,livingAreaUnits,basement,unparsedAddress,neighbourhoodName,closeDate,closePrice,location,postal,listing
0,https://www.honestdoor.com/listing/7766-eifert...,ck1i5lhmg0fev0791yw6iwuko,Residential,RMD,NaN,3.0,NaN,3.0,1664.00,1861.0,...,NaN,NaN,NaN,7766 EIFERT CRESCENT NW,Edgemont,2018-04-10T00:00:00.000Z,407110.0,"{""lat"": 53.4713802, ""lon"": -113.6715318}",T6M0W1,"{""airConditioning"": null, ""basement1"": ""Full, ..."
1,https://www.honestdoor.com/listing/3716-160a-a...,599258d70e39c7eaa472abb1,Residential,RF5,2.1,2.0,2.0,3.0,1158.42,1237.0,...,NaN,NaN,NaN,3716 160A AVENUE NW,Brintnell,2021-03-23T00:00:00.000Z,291500.0,"{""lat"": 53.6229477, ""lon"": -113.4003525}",T5Y3G1,"{""airConditioning"": null, ""basement1"": ""Full, ..."
2,https://www.honestdoor.com/listing/804-elderbe...,cmklru4ty01gq7g0at0wkuwvn,Residential,NaN,NaN,2.1,NaN,3.0,NaN,1792.0,...,NaN,NaN,NaN,804 Elderberry Court Nw,NaN,NaN,NaN,"{""lat"": 53.46022745, ""lon"": -113.683948}",NaN,"{""airConditioning"": null, ""basement1"": ""Full, ..."
3,https://www.honestdoor.com/listing/9909-171-av...,599258d70e39c7eaa472d8b2,Residential,RF5,NaN,2.0,NaN,3.0,NaN,1022.0,...,NaN,NaN,NaN,9909 171 AVENUE NW,Baturyn,2011-01-14T00:00:00.000Z,206000.0,"{""lat"": 53.632271, ""lon"": -113.495223}",T5X4X2,"{""airConditioning"": ""None"", ""basement1"": ""Full..."
4,https://www.honestdoor.com/listing/3230-dixon-...,cm672j31600fa6bynb5jindow,Residential,NaN,NaN,1.0,NaN,1.0,NaN,1127.0,...,NaN,NaN,NaN,3230 Dixon Way SW,Desrochers Area,NaN,NaN,"{""lat"": 53.40109275, ""lon"": -113.5585067}",NaN,"{""airConditioning"": null, ""basement1"": ""Full, ..."


# Fix Listings

In [14]:
import pandas as pd
import numpy as np

DROP_KEYS = {
    "HOAFee", "alternateURLVideoLink", "amperage", "analyticsClick",
    "bathrooms", "description", "garage", "leaseTerms",
    "liveStreamEventURL", "moreInformationLink", "numBathrooms",
    "numBathroomsPlus", "numBedrooms", "numBedroomsPlus",
    "numFireplaces", "numGarageSpaces", "parkCostMonthly",
    "virtualTourUrl", "yearBuilt", "zoning", "zoningDescription",
    "zoningType", "sqft", "sqftRange"
}

new_rows = []
L_col = []

def load_json_value(value):
    if pd.isna(value) or value == "":
        return {}

    if isinstance(value, dict):
        return value

    if isinstance(value, str):
        try:
            return json.loads(value)
        except json.JSONDecodeError:
            return {}

    return {}

for index, row in df.iterrows():
    listing_details = load_json_value(row["listing"])

    if not listing_details:
        new_rows.append({})
        continue


    # Build L_col from the first valid row only
    if not L_col:
        L_col = [
            f"L_{col}"
            for col in listing_details.keys()
            if col not in DROP_KEYS
        ]

    row_values = {}

    for col in listing_details.keys():
        if col in DROP_KEYS:
            continue
        
        row_values[f"L_{col}"] = listing_details[col]

    new_rows.append(row_values)

# Create all columns at once
listing_df = pd.DataFrame(new_rows, index=df.index)

# Ensure every column in L_col exists
listing_df = listing_df.reindex(columns=L_col)

df = pd.concat([df, listing_df], axis=1)

# Aggregation

In [15]:
def choose(*values):
    for v in values:
        if pd.notna(v):
            return v
    return None

df["bathroomsCount"] = [
    choose(a, b)
    for a, b in zip(
        df["bathroomsTotal"],
        df["bathroomsTotalEst"],
    )
]

df["bedroomsCount"] = [
    choose(a, b)
    for a, b in zip(
        df["bedroomsTotal"],
        df["bedroomsTotalEst"],
    )
]

df["houseStyle"] = [
    choose(a, b)
    for a, b in zip(
        df["houseStyle"],
        df["L_style"],
    )
]

df["livingArea"] = [
    choose(a, b)
    for a, b in zip(
        df["livingArea"],
        df["livingAreaEst"],
    )
]

df["lotSizeArea"] = [
    choose(a, b)
    for a, b in zip(
        df["lotSizeArea"],
        df["lotSizeAreaEst"],
    )
]

# df["basement"] = [
#     choose(a, b)
#     for a, b in zip(
#         df["basement"],
#         df["PL_basement"],
#     )
# ]

In [16]:
def extract_lat_lon(x):
    if pd.isna(x):
        return pd.Series([None, None])

    try:
        obj = json.loads(x)
        return pd.Series([
            obj.get("lat"),
            obj.get("lon")
        ])
    except:
        return pd.Series([None, None])

df[["lat", "lon"]] = df["location"].apply(extract_lat_lon)

In [17]:
def fireplace_to_binary(x):
    if pd.isna(x):
        return np.nan

    x = str(x).strip().lower()

    if x in ["yes"]:
        return 1

    return 0

df["fireplace"] = df["fireplace"].apply(fireplace_to_binary)

In [18]:
garage_counts = (
    df["garageSpaces"]
    .fillna("Missing")
    .astype(str)
    .str.strip()
    .replace("", "Missing")
    .value_counts()
)

garage_pct = garage_counts / garage_counts.sum() * 100

print("Garage Distribution (%):")
for value, pct in garage_pct.items():
    print(f"{value}: {pct:.2f}%")

Garage Distribution (%):
1.0: 59.02%
0.0: 27.81%
Missing: 13.16%


In [19]:
invalid_rows = df[
    ~df["Property URL"]
    .astype(str)
    .str.strip()
    .str.contains("https://", regex=False, na=False)
]

print(f"Found {len(invalid_rows)} invalid rows")

for idx, row in invalid_rows.iterrows():
    print(f"\nRow {idx}:")
    print("propertyURL =", repr(row["Property URL"]))

Found 0 invalid rows


In [20]:
clean_df = pd.DataFrame({
    "propertyUrl": df["Property URL"],
    "assessmentClass": df["assessmentClass"],
    "zoning": df["zoning"],
    "bathroomsCount": df["bathroomsCount"],
    "bedroomsCount": df["bedroomsCount"],
    "livingArea": df["livingArea"],
    "lotSizeArea": df["lotSizeArea"],
    "yearBuilt": df["yearBuiltActual"],
    "fireplace": df["fireplace"],
    "garage": df["garageSpaces"],
    "houseStyle": df["houseStyle"],
    "basement": df["basement"],
    "address": df["unparsedAddress"],
    "neighbourhoodName": df["neighbourhoodName"],
    "closeDate": df["closeDate"],
    "price": df["closePrice"],
    "lat": df["lat"],
    "lon": df["lon"],
})

# add L_col
clean_df = pd.concat(
    [
        clean_df,
        df[L_col],
    ],
    axis=1,
)

display(clean_df.head())

,propertyUrl,assessmentClass,zoning,bathroomsCount,bedroomsCount,livingArea,lotSizeArea,yearBuilt,fireplace,garage,...,L_patio,L_propertyType,L_roofMaterial,L_sewer,L_storageType,L_style,L_swimmingPool,L_viewType,L_waterSource,L_waterfront
0,https://www.honestdoor.com/listing/7766-eifert...,Residential,RMD,3.0,3.0,1664.00,259.00,2017.0,NaN,1.0,...,NaN,Single Family,Asphalt Shingles,None,None,2 Storey,None,None,None,None
1,https://www.honestdoor.com/listing/3716-160a-a...,Residential,RF5,2.1,2.0,1158.42,236.98,2004.0,NaN,1.0,...,NaN,Single Family,Asphalt Shingles,None,None,2 Storey,None,None,None,None
2,https://www.honestdoor.com/listing/804-elderbe...,Residential,NaN,2.1,3.0,1792.00,354.00,NaN,NaN,NaN,...,NaN,Single Family,Vinyl Shingles,None,None,2 Storey,None,None,None,None
3,https://www.honestdoor.com/listing/9909-171-av...,Residential,RF5,2.0,3.0,1022.00,289.00,1979.0,NaN,0.0,...,Deck,Row/Townhouse,Asphalt Shingle,None,None,2 Storey,None,None,None,None
4,https://www.honestdoor.com/listing/3230-dixon-...,Residential,NaN,1.0,1.0,1127.00,284.00,NaN,NaN,NaN,...,NaN,Single Family,Asphalt Shingles,None,None,2 Storey,None,None,None,None


In [21]:
# drop the same col as property dataset

cols_to_drop = ['L_airConditioning', 'L_businessSubType', 'L_businessType', 'L_ceilingType', 'L_centralAirConditioning', 'L_centralVac', 'L_certificationLevel', 'L_commonElementsIncluded', 'L_constructionStatus', 'L_constructionStyleSplitLevel', 'L_den', 'L_driveway', 'L_elevator', 'L_energuideRating', 'L_energyCertification', 'L_exteriorConstruction2', 'L_familyRoom', 'L_farmType', 'L_fireProtection', 'L_furnished', 'L_greenPropertyInformationStatement', 'L_handicappedEquipped', 'L_landAccessType', 'L_landDisposition', 'L_landSewer', 'L_landscapeFeatures', 'L_laundryLevel', 'L_loadingType', 'L_numDrivewaySpaces', 'L_numKitchens', 'L_numKitchensPlus', 'L_numParkingSpaces', 'L_numRooms', 'L_numRoomsPlus', 'L_patio', 'L_sewer', 'L_storageType', 'L_swimmingPool', 'L_viewType', 'L_waterSource', 'L_waterfront']

cols_to_drop.extend(["L_style", "L_livingAreaMeasurement", "L_propertyType"])


clean_df = clean_df.drop(columns=cols_to_drop, errors="ignore")
display(clean_df.head())

,propertyUrl,assessmentClass,zoning,bathroomsCount,bedroomsCount,livingArea,lotSizeArea,yearBuilt,fireplace,garage,...,lat,lon,L_basement1,L_basement2,L_exteriorConstruction1,L_extras,L_flooringType,L_foundationType,L_heating,L_roofMaterial
0,https://www.honestdoor.com/listing/7766-eifert...,Residential,RMD,3.0,3.0,1664.00,259.00,2017.0,NaN,1.0,...,53.471380,-113.671532,"Full, Unfinished",Unfinished,"Wood, Vinyl","Dishwasher-Built-In, Dryer, Garage Control, Ga...","Carpet, Laminate Flooring",Concrete Perimeter,"Forced Air-1, Natural Gas",Asphalt Shingles
1,https://www.honestdoor.com/listing/3716-160a-a...,Residential,RF5,2.1,2.0,1158.42,236.98,2004.0,NaN,1.0,...,53.622948,-113.400352,"Full, Finished",Fully Finished,"Wood, Brick, Vinyl","Dishwasher-Built-In, Dryer, Oven-Built-In, Ove...","Carpet, Ceramic Tile, Hardwood",Concrete Perimeter,"Forced Air-1, Natural Gas",Asphalt Shingles
2,https://www.honestdoor.com/listing/804-elderbe...,Residential,NaN,2.1,3.0,1792.00,354.00,NaN,NaN,NaN,...,53.460227,-113.683948,"Full, Unfinished",Unfinished,"Wood, Stone, Vinyl","Dishwasher-Built-In, Dryer, Garage Control, Ga...","Carpet, Vinyl Plank",Concrete Perimeter,"Forced Air-1, Natural Gas",Vinyl Shingles
3,https://www.honestdoor.com/listing/9909-171-av...,Residential,RF5,2.0,3.0,1022.00,289.00,1979.0,NaN,0.0,...,53.632271,-113.495223,"Full,Finished,None",NaN,"Cement Fiber Board,Wood Frame","Dryer,Electric Stove,Microwave Hood Fan,Refrig...","Carpet,Hardwood,Laminate",Poured Concrete,"Forced Air,Natural Gas",Asphalt Shingle
4,https://www.honestdoor.com/listing/3230-dixon-...,Residential,NaN,1.0,1.0,1127.00,284.00,NaN,NaN,NaN,...,53.401093,-113.558507,"Full, Unfinished",Unfinished,"Wood, Vinyl","Dishwasher-Built-In, Dryer, Garage Control, Ga...",Vinyl Plank,Concrete Perimeter,"Forced Air-1, Natural Gas",Asphalt Shingles


In [22]:
# save clean_df to csv
clean_df.to_csv("../data/clean/honestdoor_listing_details_clean.csv", index=False)